# Lab Assignment 5 - Multiclass Classification

**Course:** B.Tech - AI in Healthcare  
**Course Code:** CSET343  
**Semester:** VII

This notebook follows the tasks given in the assignment using the Dermatology dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Task 1: Data Acquisition and Exploration

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data"

feature_names = [
    "erythema", "scaling", "definite_borders", "itching",
    "koebner_phenomenon", "polygonal_papules", "follicular_papules",
    "oral_mucosal_involvement", "knee_elbow_involvement", "scalp_involvement",
    "family_history", "melanin_incontinence", "eosinophils_infiltrate",
    "PNL_infiltrate", "fibrosis_papillary_dermis", "exocytosis", "acanthosis",
    "hyperkeratosis", "parakeratosis", "clubbing_rete_ridges",
    "elongation_rete_ridges", "thinning_suprapapillary_epidermis",
    "spongiform_pustule", "munro_microabcess", "focal_hypergranulosis",
    "disappearance_granular_layer", "vacuolisation_basal_layer", "spongiosis",
    "saw_tooth_rete_ridges", "follicular_horn_plug",
    "perifollicular_parakeratosis", "inflammatory_mononuclear_infiltrate",
    "band_like_infiltrate", "age", "class"
]

df = pd.read_csv(DATA_URL, header=None)
df.columns = feature_names
df = df.replace("?", np.nan)

for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.head()

In [ ]:
feature_columns = [column for column in df.columns if column != "class"]

print("Dataset shape:", df.shape)

print("\nSummary statistics:")
display(df[feature_columns].describe().T[["mean", "50%", "std", "min", "max"]])

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
missing_before = df[feature_columns].isnull().sum().sum()

if missing_before > 0:
    df[feature_columns] = df[feature_columns].fillna(
        df[feature_columns].median()
    )

lower = df[feature_columns].quantile(0.25)
upper = df[feature_columns].quantile(0.75)
iqr = upper - lower

outlier_mask = (
    (df[feature_columns] < (lower - 1.5 * iqr))
    | (df[feature_columns] > (upper + 1.5 * iqr))
)

print("Outlier counts by feature:")
display(outlier_mask.sum().sort_values(ascending=False))

df[feature_columns] = df[feature_columns].clip(
    lower - 1.5 * iqr,
    upper + 1.5 * iqr,
    axis="columns",
)

In [ ]:
print("Class distribution:")
display(df["class"].value_counts().sort_index())

print("Class distribution (%):")
display((df["class"].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
df[feature_columns[:8]].hist(figsize=(12, 8))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "histograms.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[feature_columns[:10]])
plt.xticks(rotation=45)
plt.title("Box Plots of Key Features")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplots.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df[feature_columns].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="class")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150)
plt.show()

## Task 2: Data Preprocessing

In [ ]:
X = df[feature_columns]
y = df["class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## Task 3: Multiclass Model Training and Evaluation

In [ ]:
models = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000)),
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=42)),
    ]),
}

results = []
classes = np.sort(y.unique())
y_test_binary = label_binarize(y_test, classes=classes)

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )

    auc_scores = []
    for i in range(len(classes)):
        fpr, tpr, _ = roc_curve(y_test_binary[:, i], y_prob[:, i])
        auc_scores.append(auc(fpr, tpr))

    mean_auc = np.mean(auc_scores)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": mean_auc,
    })

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))
    print("ROC-AUC:", round(mean_auc, 4))

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"confusion_matrix_{name.lower().replace(' ', '_')}.png",
        dpi=150,
    )
    plt.show()

    plt.figure(figsize=(8, 6))
    for i, class_value in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_binary[:, i], y_prob[:, i])
        plt.plot(
            fpr,
            tpr,
            label=f"Class {class_value} (AUC = {auc_scores[i]:.3f})"
        )

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"roc_curve_{name.lower().replace(' ', '_')}.png",
        dpi=150,
    )
    plt.show()

In [ ]:
results_df = pd.DataFrame(results)

print("Model comparison:")
display(results_df.round(4))

results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

## Interpretation

Compare the four models using the accuracy, precision, recall, F1-score, confusion matrices and ROC-AUC values produced above. For a medical classification problem, pay particular attention to recall and the class-wise results, since missed disease cases can be important.